# kbprojection Colab setup

Run these cells once at the start of a Colab session before opening one of the experiment notebooks. Edit `PROJECT_ROOT` if your Drive folder uses a different name.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Select the project folder


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kbprojection")
os.environ["KBPROJECTION_RUNTIME"] = "colab"
os.environ["KBPROJECTION_PROJECT_ROOT"] = str(PROJECT_ROOT)

assert PROJECT_ROOT.exists(), f"Project folder not found: {PROJECT_ROOT}"
%cd /content/drive/MyDrive/kbprojection


## 3. Install the local package


In [ ]:
%pip install -r requirements-colab.txt -e .


## 4. Load API keys from Colab secrets


In [ ]:
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

for key in ["OPENAI_API_KEY", "OPENROUTER_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    if os.environ.get(key):
        continue
    if userdata is None:
        continue
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value:
        os.environ[key] = value

for key in ["OPENAI_API_KEY", "OPENROUTER_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key}:", "set" if os.environ.get(key) else "not set")


## 5. Configure persistent runtime paths


In [ ]:
from kbprojection.runtime import configure_runtime

paths = configure_runtime(project_root=PROJECT_ROOT)
for name, value in paths.as_dict().items():
    print(f"{name}: {value}")


## 6. Smoke checks


In [ ]:
import kbprojection
from kbprojection import SICKLoader
from kbprojection.downloads import check_nltk

print("kbprojection version:", kbprojection.__version__)
check_nltk("punkt_tab")
check_nltk("wordnet")
loader = SICKLoader(data_dir=paths.data_dir / "sick")
loader.load(splits=["dev"])
print("SICK dev examples loaded. Runtime setup is ready.")

RUN_LANGPRO_SMOKE = False
if RUN_LANGPRO_SMOKE:
    from kbprojection import langpro_api_call
    result = langpro_api_call(["A dog is running."], "An animal is running.")
    print(result.label, result.error)
